# ETA Model Training

Trains an XGBoost regressor that predicts segment travel time (seconds) from
traffic features.  Outputs `backend/models/eta_model.pkl`.

**Features**
| Feature | Type | Notes |
|---|---|---|
| `hour_of_day` | int | 0–23 |
| `day_of_week` | int | 0=Mon … 6=Sun |
| `road_type_enc` | int | encoded from OSM highway tag |
| `segment_length_m` | float | metres |
| `speed_limit_kmh` | float | posted limit |
| `historical_avg_speed` | float | km/h, from Uber Movement or synthetic |

**Target**: `travel_time_seconds`

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.preprocessing import LabelEncoder

SEED = 42
np.random.seed(SEED)

## 1 — Generate synthetic training data

Replace this block with real Uber Movement CSVs if available.

In [ ]:
N = 50_000

ROAD_TYPES = {
    'motorway':     {'speed_range': (80, 120), 'length_range': (500,  5000)},
    'trunk':        {'speed_range': (60,  90), 'length_range': (300,  3000)},
    'primary':      {'speed_range': (40,  70), 'length_range': (200,  2000)},
    'secondary':    {'speed_range': (30,  60), 'length_range': (100,  1500)},
    'tertiary':     {'speed_range': (20,  50), 'length_range': ( 80,  1000)},
    'residential':  {'speed_range': (15,  40), 'length_range': ( 50,   500)},
    'unclassified': {'speed_range': (15,  30), 'length_range': ( 50,   300)},
    'service':      {'speed_range': (10,  20), 'length_range': ( 20,   200)},
}
ROAD_TYPE_ENC = {t: i for i, t in enumerate(ROAD_TYPES)}

# Peak-hour congestion multipliers (higher = slower)
def congestion_multiplier(hour, dow):
    is_weekday = dow < 5
    am_peak = is_weekday and (7 <= hour <= 9)
    pm_peak = is_weekday and (16 <= hour <= 19)
    if am_peak or pm_peak:
        return np.random.uniform(1.5, 2.5)
    elif hour < 6 or hour > 22:
        return np.random.uniform(0.9, 1.1)   # light traffic
    else:
        return np.random.uniform(1.0, 1.5)

rows = []
road_type_names = list(ROAD_TYPES.keys())

for _ in range(N):
    hour   = np.random.randint(0, 24)
    dow    = np.random.randint(0, 7)
    rt     = np.random.choice(road_type_names)
    cfg    = ROAD_TYPES[rt]

    speed_limit = np.random.uniform(*cfg['speed_range'])
    length      = np.random.uniform(*cfg['length_range'])
    mult        = congestion_multiplier(hour, dow)
    avg_speed   = speed_limit / mult + np.random.normal(0, 2)
    avg_speed   = max(avg_speed, 5.0)

    travel_time = (length / 1000) / avg_speed * 3600   # seconds
    travel_time = max(travel_time + np.random.normal(0, travel_time * 0.05), 1.0)

    rows.append({
        'hour_of_day':          hour,
        'day_of_week':          dow,
        'road_type':            rt,
        'road_type_enc':        ROAD_TYPE_ENC[rt],
        'segment_length_m':     length,
        'speed_limit_kmh':      speed_limit,
        'historical_avg_speed': avg_speed,
        'travel_time_seconds':  travel_time,
    })

df = pd.DataFrame(rows)
print(df.shape)
df.head()

## 2 — Exploratory stats

In [ ]:
print(df['travel_time_seconds'].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df.groupby('hour_of_day')['historical_avg_speed'].mean().plot(
    ax=axes[0], title='Avg speed by hour', xlabel='Hour', ylabel='km/h'
)
df.groupby('road_type')['travel_time_seconds'].mean().sort_values().plot(
    kind='barh', ax=axes[1], title='Mean travel time by road type'
)
plt.tight_layout()
plt.show()

## 3 — Train / test split

In [ ]:
FEATURES = [
    'hour_of_day', 'day_of_week', 'road_type_enc',
    'segment_length_m', 'speed_limit_kmh', 'historical_avg_speed',
]
TARGET = 'travel_time_seconds'

X = df[FEATURES].values.astype(np.float32)
y = df[TARGET].values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED
)
print(f'Train: {X_train.shape}   Test: {X_test.shape}')

## 4 — Train XGBoost

In [ ]:
model = XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    random_state=SEED,
    n_jobs=-1,
    verbosity=1,
)
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50,
)
print('Done.')

## 5 — Evaluate

In [ ]:
y_pred = model.predict(X_test)
mae    = mean_absolute_error(y_test, y_pred)
rmse   = root_mean_squared_error(y_test, y_pred)
print(f'MAE:  {mae:.2f} s')
print(f'RMSE: {rmse:.2f} s')

# Feature importances
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
importances.plot(kind='barh', title='Feature importances', figsize=(8, 4))
plt.tight_layout()
plt.show()

# Residual plot
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(y_test[:2000], y_pred[:2000], alpha=0.3, s=5)
ax.plot([0, y_test.max()], [0, y_test.max()], 'r--', lw=1)
ax.set_xlabel('Actual (s)')
ax.set_ylabel('Predicted (s)')
ax.set_title('Actual vs Predicted travel time')
plt.tight_layout()
plt.show()

## 6 — Save model

In [ ]:
out_path = os.path.join('..', 'backend', 'models', 'eta_model.pkl')
os.makedirs(os.path.dirname(out_path), exist_ok=True)

with open(out_path, 'wb') as fh:
    pickle.dump(model, fh)

print(f'Model saved → {out_path}')

# Also save a small sample CSV for reference
csv_path = os.path.join('..', 'backend', 'data', 'traffic_data.csv')
df.sample(500, random_state=SEED).to_csv(csv_path, index=False)
print(f'Sample data saved → {csv_path}')